# Integrated Information Decomposition of a coordination process

This notebook applies `phyid` to **coordination dynamics** — interacting agents (think a worker, a
mediator/system, and a counterpart) whose joint behaviour unfolds over time. ΦID is usually shown on
neural or synthetic signals; here we use it to ask a question from the study of teams, markets and
platforms: *when two parties coordinate, is the information that binds them redundant, transferred,
or synergistic?*

We build three small discrete coordination regimes and read off their ΦID signatures:

| Regime | How the parties coordinate | Expected ΦID signature |
|---|---|---|
| Imitative | both copy a shared cue | **redundancy** |
| Leader–follower | follower tracks the leader's past | **directed transfer** (unique to the source) |
| Complementary | roles must differ (joint/XOR dynamics) | **synergy**, ~no redundancy |

Everything below uses only `numpy` and `phyid`. Run it top to bottom.

In [1]:
import numpy as np
from phyid.calculate import calc_PhiID, PhiID_atoms_abbr

rng = np.random.default_rng(0)
N = 30000        # series length
P_NOISE = 0.05   # per-step action noise


def phiid_means(src, trg, tau=1, redundancy="CCS"):
    # Run ΦID on two binary series; time-average the 16 atoms. (kind="discrete"
    # binarizes each series at its mean, so categorical input collapses to binary.)
    atoms, _ = calc_PhiID(src.astype(float), trg.astype(float),
                          tau=tau, kind="discrete", redundancy=redundancy)
    return {k: float(np.mean(v)) for k, v in atoms.items()}


def flip(bit):
    # Apply per-step action noise to a binary value.
    return bit if rng.random() > P_NOISE else 1 - bit

## A note on the atoms

`calc_PhiID(src, trg, ...)` returns the 16 ΦID atoms, each labelled by *(past information → future
information)* with `r` = redundant, `x` = unique to `src`, `y` = unique to `trg`, `s` = synergistic.
A few are enough for a coordination reading:

- **`rtr`** — redundancy carried forward (both parties already share it),
- **`xtr` / `x…`** — information unique to the source (one-way transfer); `xty` (unique to `src` →
  unique to `trg`) is the cleanest one-way transfer atom,
- **`sts`** — persistent synergy (information the parties only have *together*),
- **`st*`** = `str + stx + sty + sts` — the synergistic-source row: everything the two pasts carry
  only *together*, wherever it lands in the future.

With `kind="discrete"` the atoms are in bits.

## Regime 1 — Imitative coordination (a shared cue)

The worker and counterpart each copy a common, slowly-drifting system cue. They end up carrying the
*same* information, so ΦID should load on **redundancy**.

In [2]:
cue = np.zeros(N, dtype=int)
for t in range(1, N):
    cue[t] = cue[t - 1] if rng.random() > 0.1 else 1 - cue[t - 1]

worker      = np.array([flip(c) for c in cue])
counterpart = np.array([flip(c) for c in cue])

imitative = phiid_means(worker, counterpart)
print(f"redundancy (rtr) = {imitative['rtr']:+.4f}")
print(f"synergy    (sts) = {imitative['sts']:+.4f}")

redundancy (rtr) = +0.3627
synergy    (sts) = -0.0156


## Regime 2 — Leader–follower (directed coordination)

The follower tracks the leader's *previous* move. Information flows one way, so the decomposition
should expose structure that is **unique to the source** (the leader's past).

In [3]:
leader   = np.zeros(N, dtype=int)
follower = np.zeros(N, dtype=int)
for t in range(1, N):
    leader[t]   = leader[t - 1] if rng.random() > 0.15 else 1 - leader[t - 1]
    follower[t] = flip(leader[t - 1])

directed = phiid_means(leader, follower)
print(f"unique-to-leader (xtr) = {directed['xtr']:+.4f}")
print(f"redundancy       (rtr) = {directed['rtr']:+.4f}")

unique-to-leader (xtr) = +0.2591
redundancy       (rtr) = +0.3063


## Regime 3 — Complementary coordination (joint determination)

Now the parties must *differ*: the next joint state depends mostly on the **XOR** of both pasts, which
neither party's past predicts alone. (The counterpart also copies the worker's previous move half the
time, so the worker's past does carry some unique information: `xty` ≈ 0.18, `xtr` ≈ 0.04.)
The tell-tale sign of a genuinely joint determination is that the **redundancy collapses to ~zero**
— there is no longer shared information carried by each party separately; what predicts the future
lives in the pair jointly.

That joint information shows up in the **synergistic-source row** (`st*`), not in `sts`: `str` ≈ 0.19
and `stx` ≈ 0.33, for a row sum of ≈ 0.45 bits. `sts` stays slightly negative because the XOR, once it
reaches the future, is carried by each future on its own — by both (`str`, when the counterpart plays
the complement) or by the worker's alone (`stx`, when the counterpart copies) — rather than only by the
future pair together.

In [4]:
w = np.zeros(N, dtype=int)
c = np.zeros(N, dtype=int)
for t in range(1, N):
    joint = w[t - 1] ^ c[t - 1]
    w[t] = flip(joint)
    c[t] = flip(1 - joint if rng.random() > 0.5 else w[t - 1])

complementary = phiid_means(w, c)
print(f"redundancy (rtr) = {complementary['rtr']:+.4f}")
print(f"synergy    (sts) = {complementary['sts']:+.4f}")
st_row = {k: complementary[k] for k in ("str", "stx", "sty", "sts")}
print("synergistic-source row:", "  ".join(f"{k} = {v:+.4f}" for k, v in st_row.items()))
print(f"synergy past (st*) = {sum(st_row.values()):+.4f} bits")

redundancy (rtr) = +0.0000
synergy    (sts) = -0.0181
synergistic-source row: str = +0.1947  stx = +0.3306  sty = -0.0563  sts = -0.0181
synergy past (st*) = +0.4508 bits


## Summary

The three coordination regimes leave distinct ΦID fingerprints (see the table the last cell prints):

```
                       redundancy(rtr)   unique-source(xtr)   synergy-past(st*)
imitative (shared cue)     high (0.36)        ~0   (0.00)          low      (0.09)
leader -> follower         high (0.31)        high (0.26)          moderate (0.22)
complementary (joint/XOR)  ~0   (0.00)        low  (0.04)          high     (0.45)
```

So ΦID separates *how* a pair coordinates, not just *that* they are dependent: imitation shows up as
redundancy, a lead-lag relationship as one-way transfer (information unique to the leader's past),
and a joint/complementary determination as the **collapse of redundancy** plus a large
synergistic-source row — the information that binds the parties is no longer carried by either alone. This makes ΦID a natural tool for
decomposing interaction in teams, conversations, markets, and other multi-agent coordination data,
where the synergy-vs-redundancy distinction is exactly the question of interest.

The leader–follower pair also carries more redundancy than transfer (`rtr` 0.31 > `xtr` 0.26): the
leader keeps its move 85% of the time and the follower copies it, so both parties' pasts share it.
For the same reason `xty`, the cleanest one-way transfer atom, stays ≈ 0 here (0.006): what the
leader transfers also persists in the leader's own future, so ΦID files it under `xtr` (unique →
redundant). Without that persistence (a leader that picks each move at random) the transfer lands in `xty`
instead.

In [5]:
# All three regimes side by side
rows = {"imitative": imitative, "leader_follower": directed, "complementary": complementary}
print(f"{'regime':<18}{'rtr':>10}{'xtr':>10}{'xty':>10}{'sts':>10}{'st*':>10}")
for name, m in rows.items():
    st_row = m['str'] + m['stx'] + m['sty'] + m['sts']
    print(f"{name:<18}{m['rtr']:>10.4f}{m['xtr']:>10.4f}{m['xty']:>10.4f}"
          f"{m['sts']:>10.4f}{st_row:>10.4f}")

regime                   rtr       xtr       xty       sts       st*
imitative             0.3627    0.0020   -0.0326   -0.0156    0.0868
leader_follower       0.3063    0.2591    0.0062   -0.0065    0.2193
complementary         0.0000    0.0418    0.1771   -0.0181    0.4508
